# Notebook 04: Evaluation — Retrieval Metrics + LLM-Judged Correctness

Reads data/pipeline_results.json from Notebook 03 and produces the
final RAG-vs-CAG comparison for the thesis, in two parts:

  1. RETRIEVAL QUALITY (RAG only): hit@1/3/5, MRR, NDCG@5, computed
     from retrieved_article_ids vs. source_article_id. Same metric
     family as JobAI's 11_evaluate_rag.ipynb. CAG has no retrieval
     step, so this half doesn't apply to it — that's expected, not
     a gap, and is noted in the final summary.

  2. ANSWER CORRECTNESS (both pipelines): Gemini acts as an LLM
     judge, shown the question, the reference answer from
     qa_dataset.json, and the pipeline's generated answer, and
     decides if they mean the same thing (not exact string match,
     since wording legitimately varies). This is the metric that
     actually lets RAG and CAG be compared head-to-head, since it's
     architecture-agnostic.

RESUMABLE: judge verdicts are appended to data/judge_progress.jsonl
keyed by (qa_id, batch, pipeline), and already-judged triples are
skipped on re-run — same pattern as Notebook 02's Gemini calls.

Output: data/evaluation_results.json (per-record judge verdicts +
retrieval metrics) and a printed summary table comparing RAG vs CAG
at each weekly batch size.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/thesis-rag-cag')

Mounted at /content/drive


## Cell 1: Install dependencies

In [3]:
!pip install google-genai -q

## Cell 2: Imports

In [4]:
import os
import json
import time
import math
import random

from google import genai
from google.genai import types
from google.colab import userdata

## Cell 3: Config

In [5]:
API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=API_KEY)

MODEL = "gemini-3.1-flash-lite"  # same model used in Notebook 02; swap here too if it
                                  # starts hitting repeated 503s (see Notebook 02's note)

DATA_DIR = "data"
RESULTS_PATH = os.path.join(DATA_DIR, "pipeline_results.json")
JUDGE_PROGRESS_PATH = os.path.join(DATA_DIR, "judge_progress.jsonl")
OUTPUT_PATH = os.path.join(DATA_DIR, "evaluation_results.json")

BATCH_NAMES = ["week1", "week2", "week3"]
RAG_K_VALUES = [1, 3, 5]
NDCG_K = 5

MAX_RETRIES = 20
MAX_BACKOFF_SECONDS = 120
BASE_BACKOFF_SECONDS = 3
MIN_SECONDS_BETWEEN_CALLS = 4.5  # same free-tier pacing as Notebook 02
_last_call_time = 0.0

JUDGE_SCHEMA = {
    "type": "object",
    "properties": {
        "correct": {"type": "boolean"},
        "reason": {"type": "string"},
    },
    "required": ["correct", "reason"],
}

JUDGE_PROMPT = """You are grading whether a generated answer is correct, given a reference \
answer. The generated answer does not need to match the reference answer's wording — only \
its meaning. Mark "correct": false if the generated answer contradicts the reference, is \
missing the key fact the reference gives, says the information isn't available when the \
reference shows it is, or adds a claim the reference doesn't support.

Question: {question}

Reference answer: {reference_answer}

Generated answer: {generated_answer}

Is the generated answer correct?
"""

## Cell 4: Load pipeline results from Notebook 03

In [6]:
with open(RESULTS_PATH) as f:
    all_records = json.load(f)
print(f"Loaded {len(all_records)} pipeline results.")

rag_records = [r for r in all_records if r["pipeline"] == "rag"]
cag_records = [r for r in all_records if r["pipeline"] == "cag"]
print(f"  RAG: {len(rag_records)}, CAG: {len(cag_records)}")

Loaded 1718 pipeline results.
  RAG: 859, CAG: 859


## Cell 5: Retrieval-quality metrics (RAG only)

In [7]:
def hit_at_k(record, k):
    return record["source_article_id"] in record["retrieved_article_ids"][:k]


def reciprocal_rank(record):
    ids = record["retrieved_article_ids"]
    if record["source_article_id"] in ids:
        rank = ids.index(record["source_article_id"]) + 1  # 1-indexed
        return 1.0 / rank
    return 0.0


def ndcg_at_k(record, k):
    ids = record["retrieved_article_ids"][:k]
    if record["source_article_id"] not in ids:
        return 0.0
    rank = ids.index(record["source_article_id"]) + 1  # 1-indexed
    # Binary relevance, single relevant doc: DCG = 1 / log2(rank + 1),
    # IDCG = 1 / log2(1 + 1) (best case: relevant doc at rank 1).
    dcg = 1.0 / math.log2(rank + 1)
    idcg = 1.0 / math.log2(2)
    return dcg / idcg


def retrieval_metrics_for_batch(batch_records):
    n = len(batch_records)
    if n == 0:
        return None
    metrics = {f"hit@{k}": sum(hit_at_k(r, k) for r in batch_records) / n for k in RAG_K_VALUES}
    metrics["mrr"] = sum(reciprocal_rank(r) for r in batch_records) / n
    metrics[f"ndcg@{NDCG_K}"] = sum(ndcg_at_k(r, NDCG_K) for r in batch_records) / n
    return metrics


retrieval_results = {}
for batch_name in BATCH_NAMES:
    batch_rag = [r for r in rag_records if r["batch"] == batch_name]
    retrieval_results[batch_name] = retrieval_metrics_for_batch(batch_rag)

print("\n--- RAG retrieval quality (source article findable in top-k) ---")
for batch_name, metrics in retrieval_results.items():
    if metrics:
        print(f"{batch_name}: " + ", ".join(f"{k}={v:.3f}" for k, v in metrics.items()))


# ============================================================
# Cell 6: Gemini judge call helper — paced + retried (same pattern as
# Notebook 02's call_gemini)
# ============================================================

class QuotaExhausted(Exception):
    pass


def call_gemini(prompt, schema):
    global _last_call_time

    for attempt in range(MAX_RETRIES):
        elapsed = time.time() - _last_call_time
        if elapsed < MIN_SECONDS_BETWEEN_CALLS:
            time.sleep(MIN_SECONDS_BETWEEN_CALLS - elapsed)

        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=schema,
                    temperature=0.0,  # judging should be as consistent as possible
                ),
            )
            _last_call_time = time.time()
            return json.loads(response.text)
        except Exception as e:
            _last_call_time = time.time()
            error_text = str(e)
            if "RESOURCE_EXHAUSTED" in error_text and "PerDay" in error_text:
                print(f"    Daily quota exhausted for model '{MODEL}': {error_text}")
                raise QuotaExhausted(error_text)
            wait = min(BASE_BACKOFF_SECONDS * (2 ** attempt), MAX_BACKOFF_SECONDS) + random.uniform(0, 1)
            print(f"    Gemini call failed ({e}); retrying in {wait:.1f}s "
                  f"[attempt {attempt + 1}/{MAX_RETRIES}]")
            time.sleep(wait)

    print("    Giving up on this call after max retries (non-quota error).")
    return None


--- RAG retrieval quality (source article findable in top-k) ---
week1: hit@1=0.852, hit@3=0.963, hit@5=0.963, mrr=0.904, ndcg@5=0.920
week2: hit@1=0.881, hit@3=0.978, hit@5=0.978, mrr=0.927, ndcg@5=0.940
week3: hit@1=0.858, hit@3=0.958, hit@5=0.969, mrr=0.904, ndcg@5=0.921


## Cell 6: Resume support for judging

In [8]:
def load_judge_progress():
    verdicts = {}
    if os.path.exists(JUDGE_PROGRESS_PATH):
        with open(JUDGE_PROGRESS_PATH) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                record = json.loads(line)
                key = (record["qa_id"], record["batch"], record["pipeline"])
                verdicts[key] = record
    return verdicts


judge_verdicts = load_judge_progress()
if judge_verdicts:
    print(f"Resuming: {len(judge_verdicts)} judge verdicts already done.")

judge_file = open(JUDGE_PROGRESS_PATH, "a")


def record_verdict(record):
    judge_file.write(json.dumps(record) + "\n")
    judge_file.flush()
    os.fsync(judge_file.fileno())

Resuming: 1478 judge verdicts already done.


## Cell 7: Run the judge over every record (both pipelines)

In [9]:
stopped_early = False
to_judge = [
    r for r in all_records
    if (r["qa_id"], r["batch"], r["pipeline"]) not in judge_verdicts
]
print(f"\n{len(to_judge)} records left to judge ({len(judge_verdicts)} already done).")

for i, record in enumerate(to_judge):
    key = (record["qa_id"], record["batch"], record["pipeline"])
    prompt = JUDGE_PROMPT.format(
        question=record["question"],
        reference_answer=record["reference_answer"],
        generated_answer=record["generated_answer"],
    )
    try:
        verdict = call_gemini(prompt, JUDGE_SCHEMA)
    except QuotaExhausted:
        stopped_early = True
        break

    if verdict is None:
        print("    Stopping run: repeated non-quota API failures. "
              "Re-run this notebook to continue from here.")
        stopped_early = True
        break

    record_verdict({
        "qa_id": record["qa_id"],
        "batch": record["batch"],
        "pipeline": record["pipeline"],
        "correct": verdict["correct"],
        "reason": verdict["reason"],
    })

    if (i + 1) % 25 == 0:
        print(f"  {i + 1}/{len(to_judge)} judged")

judge_file.close()

if stopped_early:
    print("\n*** Stopped early on a quota limit — re-run this notebook later to continue. ***")
else:
    print("\nAll records judged.")


240 records left to judge (1478 already done).
  25/240 judged
  50/240 judged
  75/240 judged
  100/240 judged
  125/240 judged
  150/240 judged
  175/240 judged
  200/240 judged
  225/240 judged

All records judged.


## Cell 8: Merge verdicts into results, compute final comparison

In [10]:
judge_verdicts = load_judge_progress()  # reload fresh, includes this run's writes

for record in all_records:
    key = (record["qa_id"], record["batch"], record["pipeline"])
    verdict = judge_verdicts.get(key)
    record["judged_correct"] = verdict["correct"] if verdict else None
    record["judge_reason"] = verdict["reason"] if verdict else None

with open(OUTPUT_PATH, "w") as f:
    json.dump(all_records, f, indent=2)
print(f"\nSaved merged evaluation results to {OUTPUT_PATH}")


def correctness_rate(records, question_type=None):
    filtered = [r for r in records if r["judged_correct"] is not None]
    if question_type:
        filtered = [r for r in filtered if r["question_type"] == question_type]
    if not filtered:
        return None
    return sum(1 for r in filtered if r["judged_correct"]) / len(filtered)


Saved merged evaluation results to data/evaluation_results.json


## Cell 9: Final summary table

In [11]:
print("\n" + "=" * 70)
print("FINAL COMPARISON: RAG vs CAG, by weekly batch size")
print("=" * 70)

for batch_name in BATCH_NAMES:
    batch_rag = [r for r in all_records if r["batch"] == batch_name and r["pipeline"] == "rag"]
    batch_cag = [r for r in all_records if r["batch"] == batch_name and r["pipeline"] == "cag"]

    print(f"\n--- {batch_name} ---")

    rag_overall = correctness_rate(batch_rag)
    rag_factual = correctness_rate(batch_rag, "factual")
    rag_reasoning = correctness_rate(batch_rag, "reasoning")
    print(f"RAG answer correctness: overall={rag_overall:.2%}  "
          f"factual={rag_factual:.2%}  reasoning={rag_reasoning:.2%}  (n={len(batch_rag)})")
    if retrieval_results.get(batch_name):
        print(f"RAG retrieval quality: " +
              ", ".join(f"{k}={v:.3f}" for k, v in retrieval_results[batch_name].items()))

    cag_overall = correctness_rate(batch_cag)
    cag_factual = correctness_rate(batch_cag, "factual")
    cag_reasoning = correctness_rate(batch_cag, "reasoning")
    print(f"CAG answer correctness: overall={cag_overall:.2%}  "
          f"factual={cag_factual:.2%}  reasoning={cag_reasoning:.2%}  (n={len(batch_cag)})")
    if batch_cag:
        avg_tokens = sum(r["context_token_count"] for r in batch_cag) / len(batch_cag)
        source_included_rate = sum(r["source_article_included"] for r in batch_cag) / len(batch_cag)
        print(f"CAG context: avg {avg_tokens:.0f} tokens/call, "
              f"source article actually in context {source_included_rate:.2%} of the time")
        if source_included_rate < 1.0:
            print("  ^ some of CAG's apparent incorrectness at this batch size is really "
                  "context truncation dropping the source article before generation even "
                  "runs — worth reporting separately from genuine reasoning failures.")

print("\nDone. This is the core result set for the thesis's RAG-vs-CAG comparison.")


FINAL COMPARISON: RAG vs CAG, by weekly batch size

--- week1 ---
RAG answer correctness: overall=88.89%  factual=85.19%  reasoning=92.59%  (n=54)
RAG retrieval quality: hit@1=0.852, hit@3=0.963, hit@5=0.963, mrr=0.904, ndcg@5=0.920
CAG answer correctness: overall=92.59%  factual=96.30%  reasoning=88.89%  (n=54)
CAG context: avg 3333 tokens/call, source article actually in context 100.00% of the time

--- week2 ---
RAG answer correctness: overall=91.85%  factual=92.31%  reasoning=91.43%  (n=135)
RAG retrieval quality: hit@1=0.881, hit@3=0.978, hit@5=0.978, mrr=0.927, ndcg@5=0.940
CAG answer correctness: overall=55.56%  factual=61.54%  reasoning=50.00%  (n=135)
CAG context: avg 4991 tokens/call, source article actually in context 59.26% of the time
  ^ some of CAG's apparent incorrectness at this batch size is really context truncation dropping the source article before generation even runs — worth reporting separately from genuine reasoning failures.

--- week3 ---
RAG answer correctn